In [ ]:
#-------------------------------------------
# Step 1: Download the Dataset from Kaggle
#-------------------------------------------

In [ ]:
from google.colab import files
import os

# 1. Trigger the file upload dialog and select your downloaded kaggle.json file
uploaded = files.upload()

# 2. Move the uploaded key to the hidden ~/.kaggle directory and set file permissions
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 3. Download the dataset via Kaggle API
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia

# 4. Extract the images into the folder 'chest_xray_data'
!unzip -q chest-xray-pneumonia.zip -d chest_xray_data

print("Dataset downloaded and unzipped successfully!")

In [ ]:
#-------------------------------------------
# Step 2: Data Preprocessing & Augmentation
#-------------------------------------------

In [2]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# Define data transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load Datasets
train_dir = 'chest_xray_data/chest_xray/train'
test_dir = 'chest_xray_data/chest_xray/test'

train_dataset = ImageFolder(root=train_dir, transform=train_transform)
test_dataset = ImageFolder(root=test_dir, transform=val_test_transform)

# Create Data Loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Classes: {train_dataset.classes}")

Classes: ['NORMAL', 'PNEUMONIA']


In [ ]:
#-------------------------------------------
# Step 3: Build the Model (Transfer Learning)
#-------------------------------------------

In [3]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

# Load pre-trained ResNet-18
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

# Replace the final classification head for 2 output classes (NORMAL vs PNEUMONIA)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)

# Move model to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 165MB/s]


In [ ]:
#-------------------------------------------
# Step 4: Define Loss Function & Optimizer
#-------------------------------------------

In [5]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
#-------------------------------------------
# Step 5: Train and Evaluate the Model
#-------------------------------------------

In [12]:



num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {epoch_loss:.4f} - Train Acc: {epoch_acc*100:.2f}%")

# Test evaluation
model.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        test_correct += (preds == labels).sum().item()
        test_total += labels.size(0)

print(f"\nFinal Test Accuracy: {(test_correct / test_total) * 100:.2f}%")


Epoch [1/10] - Train Loss: 0.0254 - Train Acc: 99.14%
Epoch [2/10] - Train Loss: 0.0209 - Train Acc: 99.25%
Epoch [3/10] - Train Loss: 0.0198 - Train Acc: 99.35%
Epoch [4/10] - Train Loss: 0.0243 - Train Acc: 99.21%
Epoch [5/10] - Train Loss: 0.0175 - Train Acc: 99.46%
Epoch [6/10] - Train Loss: 0.0136 - Train Acc: 99.56%
Epoch [7/10] - Train Loss: 0.0211 - Train Acc: 99.33%
Epoch [8/10] - Train Loss: 0.0101 - Train Acc: 99.64%
Epoch [9/10] - Train Loss: 0.0165 - Train Acc: 99.37%
Epoch [10/10] - Train Loss: 0.0147 - Train Acc: 99.50%

Final Test Accuracy: 84.13%


In [ ]:
#-------------------------------------------
# Step 6: Save Model Checkpoint
#-------------------------------------------

In [ ]:
torch.save(model.state_dict(), 'pneumonia_resnet18.pth')
print("Model saved successfully!")

In [ ]:
#-------------------------------------------
# Step 7: Check it on Real Image from the internet
#-------------------------------------------

In [14]:
import torch
import torchvision.transforms as transforms
from PIL import Image
from google.colab import files
import io

# 1. Upload an image file directly from your computer
print("Please upload a chest X-ray image:")
uploaded = files.upload()

# Get the uploaded filename
file_name = list(uploaded.keys())[0]

# 2. Define the exact same preprocessing transform used for testing
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Load and preprocess the uploaded image
image = Image.open(io.BytesIO(uploaded[file_name])).convert('RGB')
input_tensor = inference_transform(image).unsqueeze(0)  # Add batch dimension: [1, 3, 224, 224]
input_tensor = input_tensor.to(device)

# 4. Perform Inference
model.eval()
with torch.no_grad():
    outputs = model(input_tensor)
    probabilities = torch.softmax(outputs, dim=1)
    confidence, predicted_class = torch.max(probabilities, 1)

# 5. Output prediction results
class_names = train_dataset.classes  # ['NORMAL', 'PNEUMONIA']
predicted_label = class_names[predicted_class.item()]
confidence_score = confidence.item() * 100

print(f"\nUploaded File: {file_name}")
print(f"Prediction: {predicted_label}")
print(f"Confidence: {confidence_score:.2f}%")

Please upload a chest X-ray image:


KeyboardInterrupt: 

In [ ]:
#-------------------------------------------
# Improving the Model by doing some techeniques The First Try (1)
#-------------------------------------------

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision.models import densenet121, DenseNet121_Weights
import numpy as np

# 1. Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Enhanced Data Augmentation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Load Datasets & DataLoaders
train_dir = 'chest_xray_data/chest_xray/train'
test_dir = 'chest_xray_data/chest_xray/test'

train_dataset = ImageFolder(root=train_dir, transform=train_transform)
test_dataset = ImageFolder(root=test_dir, transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 4. Calculate Class Weights to handle dataset imbalance
targets = [label for _, label in train_dataset.samples]
class_counts = np.bincount(targets)
class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
class_weights = class_weights / class_weights.sum()  # Normalize weights
class_weights = class_weights.to(device)

print(f"Class distribution (Normal / Pneumonia): {class_counts}")
print(f"Calculated loss weights: {class_weights}")

# 5. Load DenseNet-121 (Gold standard for Chest X-Rays)
weights = DenseNet121_Weights.DEFAULT
model = densenet121(weights=weights)

# Freeze early layers to prevent overfitting on small data
for param in list(model.parameters())[:-30]:
    param.requires_grad = False

# Custom classifier head with Dropout
num_features = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(num_features, 2)
)
model = model.to(device)

# 6. Weighted Loss, Optimizer with L2 Regularization, and Scheduler
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# 7. Training Loop with Best-Model Saving
num_epochs = 10
best_acc = 0.0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    scheduler.step()
    train_loss = running_loss / total
    train_acc = correct / total

    # Evaluate on Test set after each epoch
    model.eval()
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            test_correct += (preds == labels).sum().item()
            test_total += labels.size(0)

    test_acc = test_correct / test_total
    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f} - Train Acc: {train_acc*100:.2f}% | Test Acc: {test_acc*100:.2f}%")

    # Save checkpoint if test accuracy improves
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), 'best_pneumonia_densenet.pth')

print(f"\nTraining Complete! Highest Test Accuracy achieved: {best_acc*100:.2f}%")

Class distribution (Normal / Pneumonia): [1341 3875]
Calculated loss weights: tensor([0.7429, 0.2571], device='cuda:0')
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 169MB/s]


Epoch [1/10] - Train Loss: 0.2209 - Train Acc: 90.68% | Test Acc: 87.50%
Epoch [2/10] - Train Loss: 0.1109 - Train Acc: 95.74% | Test Acc: 88.46%
Epoch [3/10] - Train Loss: 0.0867 - Train Acc: 96.91% | Test Acc: 88.94%
Epoch [4/10] - Train Loss: 0.0729 - Train Acc: 97.26% | Test Acc: 90.06%
Epoch [5/10] - Train Loss: 0.0791 - Train Acc: 97.18% | Test Acc: 91.19%
Epoch [6/10] - Train Loss: 0.0695 - Train Acc: 97.58% | Test Acc: 87.66%
Epoch [7/10] - Train Loss: 0.0658 - Train Acc: 97.49% | Test Acc: 89.26%
Epoch [8/10] - Train Loss: 0.0640 - Train Acc: 97.68% | Test Acc: 89.10%
Epoch [9/10] - Train Loss: 0.0631 - Train Acc: 97.66% | Test Acc: 89.10%
Epoch [10/10] - Train Loss: 0.0582 - Train Acc: 97.68% | Test Acc: 89.74%

Training Complete! Highest Test Accuracy achieved: 91.19%


In [18]:
import torch
import torchvision.transforms as transforms
from PIL import Image
from google.colab import files
import io

# 1. Upload an image file directly from your computer
print("Please upload a chest X-ray image:")
uploaded = files.upload()

# Get the uploaded filename
file_name = list(uploaded.keys())[0]

# 2. Define the exact same preprocessing transform used for testing
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Load and preprocess the uploaded image
image = Image.open(io.BytesIO(uploaded[file_name])).convert('RGB')
input_tensor = inference_transform(image).unsqueeze(0)  # Add batch dimension: [1, 3, 224, 224]
input_tensor = input_tensor.to(device)

# 4. Perform Inference
model.eval()
with torch.no_grad():
    outputs = model(input_tensor)
    probabilities = torch.softmax(outputs, dim=1)
    confidence, predicted_class = torch.max(probabilities, 1)

# 5. Output prediction results
class_names = train_dataset.classes  # ['NORMAL', 'PNEUMONIA']
predicted_label = class_names[predicted_class.item()]
confidence_score = confidence.item() * 100

print(f"\nUploaded File: {file_name}")
print(f"Prediction: {predicted_label}")
print(f"Confidence: {confidence_score:.2f}%")

Please upload a chest X-ray image:


Saving images-normal.jpg to images-normal (1).jpg

Uploaded File: images-normal (1).jpg
Prediction: NORMAL
Confidence: 71.39%


In [ ]:
#-------------------------------------------
# Improving the Model by doing some techeniques The Second Try (2)
#-------------------------------------------

In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
from torchvision.models import densenet121, DenseNet121_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# 1. Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Transforms (Basic resize for dataset loading; Mixup will handle augmentation)
data_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Combine all data and perform an 80/20 Stratified Split
train_dir = 'chest_xray_data/chest_xray/train'
test_dir = 'chest_xray_data/chest_xray/test'
val_dir = 'chest_xray_data/chest_xray/val'

# Load full datasets
full_train = ImageFolder(root=train_dir, transform=data_transform)
full_test = ImageFolder(root=test_dir, transform=data_transform)
full_val = ImageFolder(root=val_dir, transform=data_transform)

# Combine all samples into one unified dataset pool
full_dataset = torch.utils.data.ConcatDataset([full_train, full_test, full_val])
targets = [s[1] for s in full_train.samples] + [s[1] for s in full_test.samples] + [s[1] for s in full_val.samples]

# Stratified Split (80% train, 20% test)
train_idx, test_idx = train_test_split(
    np.arange(len(targets)),
    test_size=0.20,
    shuffle=True,
    stratify=targets,
    random_state=42
)

train_subset = Subset(full_dataset, train_idx)
test_subset = Subset(full_dataset, test_idx)

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=32, shuffle=False)

print(f"Total images: {len(full_dataset)} | Train split: {len(train_subset)} | Test split: {len(test_subset)}")

# 4. Mixup Augmentation Helper Function
def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# 5. Build Model (DenseNet-121)
weights = DenseNet121_Weights.DEFAULT
model = densenet121(weights=weights)

for param in list(model.parameters())[:-30]:
    param.requires_grad = False

num_features = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(num_features, 2)
)
model = model.to(device)

# Loss, Optimizer, Scheduler
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# 6. Training Loop with Mixup
num_epochs = 10
best_acc = 0.0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Apply Mixup
        images, targets_a, targets_b, lam = mixup_data(images, labels, alpha=0.2)

        optimizer.zero_grad()
        outputs = model(images)
        loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (lam * (preds == targets_a).sum().item() + (1 - lam) * (preds == targets_b).sum().item())
        total += labels.size(0)

    scheduler.step()
    train_loss = running_loss / total
    train_acc = correct / total

    # Quick evaluation after epoch
    model.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            test_correct += (preds == labels).sum().item()
            test_total += labels.size(0)

    test_acc = test_correct / test_total
    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f} - Train Acc: {train_acc*100:.2f}% | Test Acc: {test_acc*100:.2f}%")

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), 'best_pneumonia_mixup.pth')

# 7. Final Comprehensive Evaluation (Recall, Precision, Confusion Matrix)
print("\n--- Running Final Evaluation on Test Set ---")
model.load_state_dict(torch.load('best_pneumonia_mixup.pth'))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# Display Classification Metrics
class_names = full_train.classes  # ['NORMAL', 'PNEUMONIA']
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

print("Confusion Matrix:")
cm = confusion_matrix(all_labels, all_preds)
print(f"TN: {cm[0][0]} | FP: {cm[0][1]}")
print(f"FN: {cm[1][0]} | TP: {cm[1][1]}")

Total images: 5856 | Train split: 4684 | Test split: 1172
Epoch [1/10] - Train Loss: 0.2920 - Train Acc: 87.99% | Test Acc: 95.14%
Epoch [2/10] - Train Loss: 0.2199 - Train Acc: 92.11% | Test Acc: 96.50%
Epoch [3/10] - Train Loss: 0.1877 - Train Acc: 93.72% | Test Acc: 96.67%
Epoch [4/10] - Train Loss: 0.1868 - Train Acc: 94.01% | Test Acc: 96.59%
Epoch [5/10] - Train Loss: 0.1640 - Train Acc: 95.30% | Test Acc: 96.50%
Epoch [6/10] - Train Loss: 0.1646 - Train Acc: 94.75% | Test Acc: 96.42%
Epoch [7/10] - Train Loss: 0.1712 - Train Acc: 94.19% | Test Acc: 96.59%
Epoch [8/10] - Train Loss: 0.1666 - Train Acc: 94.42% | Test Acc: 96.59%
Epoch [9/10] - Train Loss: 0.1569 - Train Acc: 94.86% | Test Acc: 96.67%
Epoch [10/10] - Train Loss: 0.1467 - Train Acc: 95.28% | Test Acc: 96.50%

--- Running Final Evaluation on Test Set ---

Classification Report:
              precision    recall  f1-score   support

      NORMAL       0.95      0.92      0.94       317
   PNEUMONIA       0.97      0.9

In [22]:
import torch
import torchvision.transforms as transforms
from PIL import Image
from google.colab import files
import io

# 1. Upload an image file directly from your computer
print("Please upload a chest X-ray image:")
uploaded = files.upload()

# Get the uploaded filename
file_name = list(uploaded.keys())[0]

# 2. Define the exact same preprocessing transform used for testing
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Load and preprocess the uploaded image
image = Image.open(io.BytesIO(uploaded[file_name])).convert('RGB')
input_tensor = inference_transform(image).unsqueeze(0)  # Add batch dimension: [1, 3, 224, 224]
input_tensor = input_tensor.to(device)

# 4. Perform Inference
model.eval()
with torch.no_grad():
    outputs = model(input_tensor)
    probabilities = torch.softmax(outputs, dim=1)
    confidence, predicted_class = torch.max(probabilities, 1)

# 5. Output prediction results
class_names = train_dataset.classes  # ['NORMAL', 'PNEUMONIA']
predicted_label = class_names[predicted_class.item()]
confidence_score = confidence.item() * 100

print(f"\nUploaded File: {file_name}")
print(f"Prediction: {predicted_label}")
print(f"Confidence: {confidence_score:.2f}%")

Please upload a chest X-ray image:


Saving ijap5072666-fig-0002b-m.jpg to ijap5072666-fig-0002b-m (2).jpg

Uploaded File: ijap5072666-fig-0002b-m (2).jpg
Prediction: PNEUMONIA
Confidence: 92.50%
